# xrf-raw-to-h5 — demo: CLI + UI

I tre percorsi d'uso della stessa pipeline di conversione (lettura → calibrazione →
rebin opzionale → writer): **CLI** (quella che lanceresti da terminale),
**API Python** (`xrfraw.io.convert_file`) e **UI embeddata** (Dash interattiva).

Prerequisito: eseguire dentro la venv del progetto (`uv sync --dev`).
Se `data_example/mappa11.txt` è presente viene usato; altrimenti viene
generato un piccolo file sintetico (formato B, griglia 3×2, 8 eventi).
Gli output finiscono in `demo_out/` (gitignored).

In [ ]:
import subprocess
import sys
from pathlib import Path

# Se esiste il dato d'esempio reale lo usiamo; altrimenti file sintetico piccolo (formato B)
RAW = Path("data_example/mappa11.txt")
if not RAW.is_file():
    RAW = Path("demo_tiny.txt")
    RAW.write_text("\n".join([
        # serpentina 3x2: X 0..2000 step 1000, Y 0..5000 step 5000, 8 eventi
        "50000000", "60000000", "0", "5", "5", "7",
        "50001000", "60000000",
        "50002000", "60000000", "0",
        "50000000", "60005000",
        "50001000", "60005000", "12",
        "50002000", "60005000", "12", "12",
        "</analysisData>", "</XRFanalysis>", "",
    ]))
OUT = Path("demo_out")
OUT.mkdir(exist_ok=True)
print(f"raw di demo: {RAW} ({RAW.stat().st_size / 1e6:.2f} MB)")
print(f"python: {sys.executable}")

raw di demo: data_example/mappa11.txt (99.42 MB)
python: /workspace/exchange/xrf-raw-converter/.venv/bin/python


## 1. CLI

La stessa identica command line del terminale: `xrf-raw-to-h5 INPUT -o OUT_DIR`.
Il notebook la lancia tramite `python -m xrfraw` (lo script console
`xrf-raw-to-h5` è il suo alias).

Opzioni: `--format h5|npz`, `--n-channels N` (default 16384, profondità
esatta del cubo), `--dtype uint32|uint16`, `--metadata JSON`
(griglia/calibrazione per formato B), `--rebin SIZE sum|mean`,
`--rebin-to {512,1024,2048,4096,8192}` (profondità target, solo sum),
`--compress 0-9` (default 4), `--overwrite` (default: rifiuta di sovrascrivere).
Exit code: 0 ok, 1 conversione fallita, 2 errore d'uso.


In [ ]:
# stessa command line del terminale
cmd = [sys.executable, "-m", "xrfraw", str(RAW),
       "-o", str(OUT), "--format", "h5", "--overwrite"]
print("$", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout.strip())
print(r.stderr.strip())
assert r.returncode == 0, "CLI fallita"
out_file = OUT / f"{RAW.stem}.h5"
print(f"output: {out_file} ({out_file.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import json

import h5py

with h5py.File(out_file, "r") as h:
    img  = h["img"][...]
    meta = json.loads(h.attrs["metadata_json"])   # metadata_json è un attr della root

print("shape:", img.shape, img.dtype)   # (ny, nx, n_channels)
print("n_events:", meta["n_events"], "| sum(img):", int(img.sum()),
      "| coerenti:", int(img.sum()) == meta["n_events"])
print("anomalies:", meta.get("anomalies"))

## 2. API Python

Stessa pipeline senza terminale: `xrfraw.io.convert_file` è la funzione
che CLI e UI condividono. Qui legge alla profondità strumentale (16384)
e usa `rebin_to` per ottenere il cubo finale alla profondità `n_chn`
desiderata — (y, x, 16384) → (y, x, n_chn), somma per gruppi,
totali conservati — e restituisce il `Dataset` scritto: `ds.img`
(y, x, canali) e `ds.metadata`.
Tutti i parametri sono scritti esplicitamente con il loro valore:
è la lista completa di ciò che si può cambiare.


In [ ]:
from xrfraw.io import convert_file

# n_chn: profondità finale desiderata del cubo (y, x, n_chn).
# Contratto: n_channels è la profondità ESATTA di lettura; per ridurla si
# passa rebin_to (somma per gruppi, totali conservati):
# (y, x, 16384) -> (y, x, n_chn).
n_chn = 2048
# tutti i parametri espliciti (nome=valore): è la lista completa di ciò che possiamo cambiare
ds = convert_file(
    src=RAW,                   # input raw (formato A o B)
    out_path=OUT / f"{RAW.stem}_{n_chn}.h5",   # output
    fmt="h5",                  # "h5" | "npz"
    n_channels=16384,          # profondità ESATTA di lettura (y, x, 16384); errore se un canale la supera
    dtype="uint32",            # "uint32" | "uint16" | "float32" | "float64" (float: compat dash-xrf-visualizer)
    metadata=None,             # formato B: dict griglia/calibrazione
    rebin_spec=None,           # es. (2, "sum") o (2, "mean"); None = off (mutuo esclusivo con rebin_to)
    rebin_to=n_chn,            # profondità target {512,1024,2048,4096,8192}; solo sum (totali conservati)
    cal=None,                  # None = calibrazione dichiarata nell'XML; oppure {"m": ..., "q": ...}
    compress=4,                # gzip 0-9 (0 = none); default 4
    overwrite=True,            # False = rifiuta di sovrascrivere un file esistente
)
print("shape:", ds.img.shape, ds.img.dtype)
print("n_events:", ds.metadata["n_events"], "| sum(img):", int(ds.img.sum()),
      "| coerenti:", int(ds.img.sum()) == ds.metadata["n_events"])


shape: (76, 286, 2048) uint32
n_events: 19825090 | sum(img): 19825090 | coerenti: True


## 3. UI embeddata

La stessa pipeline, interattiva: carichi il raw (upload, *load from link* o
`path_to_file`), imposti ogni parametro di `convert_file` (out_path, fmt,
n_channels, dtype, metadata, rebin, cal m/q, compress, overwrite), poi
**Convert** e **Download file** per l'output `.h5`/`.npz`. Utile per
esplorare i file prima di lanciarli in batch via CLI.

Nota: l'app inline di Dash usa la **porta fissa 8050** (per tutte le app
dash inline di questa macchina). Se un'altra app inline è ancora in
esecuzione — es. l'example di dash-xrf-visualizer — l'iframe di questa cell
mostrerebbe quella: chiudila o *Kernel → Restart* prima di eseguire questa
cell.


In [ ]:
path_to_file = RAW    # come dash-xrf-visualizer: build_app(src=...) precompila path_to_file

from xrfraw.ui import build_app

app = build_app(src=path_to_file)
app.run(
    jupyter_mode="inline", 
    debug=False, 
    jupyter_height=800
)  # da terminale: app.run()


## Note

- CLI: scrive `demo_out/<nome-input>.h5`; UI: restituisce il file con
  **Download file**.
- Rebin: sempre esplicito, off di default, dichiarato nei metadati
  dell'output; `rebin_to` mappa il cubo a profondità target
  (512–8192) con sola somma (totali conservati).
- Calibrazione: applicata automaticamente se l'XML la dichiara (formato A);
  per il formato B si imposta dalla UI (`cal m`/`cal q`) o con `--metadata`.
- Senza `--overwrite` la CLI rifiuta di sovrascrivere (stesso comportamento
  nel writer: nessun overwrite silenzioso).
